# DiffSinger Acoustic Training (Colab)

Train a DiffSinger acoustic model on your own singing voice dataset.

**Before first run:** Upload `<singer>_binary.zip` and your `<singer>_acoustic.yaml` config to Google Drive.

**Runtime:** GPU → T4 or L4. Checkpoints save to Google Drive and survive session restarts.

**Every session:** Run All cells. It automatically resumes from the last checkpoint.

## 0. Configuration
Set these variables for your singer profile.

In [ ]:
# ====== EDIT THESE ======
SINGER_NAME = 'my_singer'                # Your singer profile name
BINARY_ZIP  = f'{SINGER_NAME}_binary.zip' # Filename in Google Drive root
CONFIG_YAML = f'{SINGER_NAME}_acoustic.yaml' # Config filename in Google Drive root
# ========================

## 1. Mount Google Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = f'/content/drive/MyDrive/diffsinger_training/{SINGER_NAME}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive workspace: {DRIVE_DIR}')

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!git clone https://github.com/openvpi/DiffSinger.git /content/DiffSinger
%cd /content/DiffSinger

In [ ]:
!pip install -q lightning~=2.3.0 librosa==0.9.2 scipy>=1.10.0 pyworld==0.3.4 \
    praat-parselmouth==0.4.3 einops>=0.7.0 onnx~=1.16.0 tensorboardX \
    torchmetrics click tqdm PyYAML h5py MonkeyType==23.3.0

## 2. Upload data & download vocoder

In [ ]:
import os

binary_zip = f'/content/drive/MyDrive/{BINARY_ZIP}'
if not os.path.exists(binary_zip):
    raise FileNotFoundError(
        f'Upload {BINARY_ZIP} to your Google Drive root first!\n'
        'Go to drive.google.com and drop the file there.'
    )

!mkdir -p /content/DiffSinger/data/{SINGER_NAME}
!unzip -o "$binary_zip" -d /content/DiffSinger/data/{SINGER_NAME}/
!ls -lh /content/DiffSinger/data/{SINGER_NAME}/binary/

In [ ]:
import json, urllib.request, zipfile, os

VOCODER_DIR = '/content/DiffSinger/checkpoints/pc_nsf_hifigan_44.1k_hop512_128bin_2025.02'
if not os.path.exists(VOCODER_DIR):
    print('Downloading vocoder...')
    api_url = 'https://api.github.com/repos/openvpi/vocoders/releases/tags/pc-nsf-hifigan-44.1k-hop512-128bin-2025.02'
    with urllib.request.urlopen(api_url) as resp:
        release = json.loads(resp.read())
    asset = [a for a in release['assets']
             if 'onnx' not in a['name'].lower()
             and 'openutau' not in a['name'].lower()
             and 'oudep' not in a['name'].lower()][0]
    zip_path = f"/content/{asset['name']}"
    urllib.request.urlretrieve(asset['browser_download_url'], zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/DiffSinger/checkpoints/')
    os.remove(zip_path)
    print(f'Vocoder ready: {os.listdir(VOCODER_DIR)}')
else:
    print('Vocoder already present.')

## 3. Training config

Copies your config and dictionary from Google Drive into DiffSinger.

In [ ]:
import shutil

# Copy config from Google Drive
config_src = f'/content/drive/MyDrive/{CONFIG_YAML}'
config_dst = f'/content/DiffSinger/configs/{SINGER_NAME}_acoustic.yaml'
if os.path.exists(config_src):
    shutil.copy2(config_src, config_dst)
    print(f'Config copied: {config_dst}')
else:
    raise FileNotFoundError(f'Upload {CONFIG_YAML} to Google Drive root first!')

# Copy dictionary if present in the binary zip (check data dir)
dict_dir = '/content/DiffSinger/dictionaries'
os.makedirs(dict_dir, exist_ok=True)
# If you have a custom dictionary, upload it to Drive and copy it here:
# shutil.copy2('/content/drive/MyDrive/english_ipa.txt', f'{dict_dir}/english_ipa.txt')
print(f'Dictionaries: {os.listdir(dict_dir) if os.path.exists(dict_dir) else "(empty)"}')

## 4. Train

Checkpoints save to Google Drive. If the session dies, just start a new session and **Run All** — it resumes automatically.

In [ ]:
import os, glob

%env PYTHONPATH=/content/DiffSinger
%env TORCH_CUDNN_V8_API_ENABLED=1

EXP_NAME = f'{SINGER_NAME}_acoustic'
DRIVE_CKPT = f'/content/drive/MyDrive/diffsinger_training/{SINGER_NAME}/{EXP_NAME}'
LOCAL_CKPT = f'/content/DiffSinger/checkpoints/{EXP_NAME}'
os.makedirs(DRIVE_CKPT, exist_ok=True)
if os.path.exists(LOCAL_CKPT) and not os.path.islink(LOCAL_CKPT):
    !mv {LOCAL_CKPT}/* {DRIVE_CKPT}/ 2>/dev/null; rm -rf {LOCAL_CKPT}
if not os.path.exists(LOCAL_CKPT):
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)

existing = sorted(glob.glob(f'{DRIVE_CKPT}/model_ckpt_steps_*.ckpt'))
if existing:
    print(f'Resuming from: {os.path.basename(existing[-1])}')
    reset_flag = ''
else:
    print('Starting fresh (no previous checkpoints found)')
    reset_flag = '--reset'

!python scripts/train.py \
    --config configs/{EXP_NAME}.yaml \
    --exp_name {EXP_NAME} \
    $reset_flag

## 5. Monitor (optional)

In [ ]:
# Uncomment to launch TensorBoard
# %load_ext tensorboard
# %tensorboard --logdir /content/DiffSinger/checkpoints/{EXP_NAME}/lightning_logs/

## 6. Download checkpoint

In [ ]:
import glob, os

ckpts = sorted(glob.glob(f'{DRIVE_CKPT}/model_ckpt_steps_*.ckpt'))
print('Available checkpoints:')
for c in ckpts:
    print(f'  {os.path.basename(c)} ({os.path.getsize(c) / 1e6:.1f} MB)')

In [ ]:
!cd {DRIVE_CKPT}/.. && zip -r /content/{EXP_NAME}_ckpt.zip {EXP_NAME}/ \
    -x '{EXP_NAME}/lightning_logs/*'

from google.colab import files
files.download(f'/content/{EXP_NAME}_ckpt.zip')